# Minimization of total cost with constraints on GHG emissions for the 2050 energy system with varying regionalization levels

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from utils import *
import plotly.express as px
import bw2data as bd
import pandas as pd
from new_plots import _create_sankey_figure, generate_sankey_flows
from tqdm import tqdm
import ast
import time

In [3]:
import plotly.io as pio
pio.renderers.default = "png"

In [6]:
es_tech_df = pd.read_csv('../01_Notebooks/Data/technology_dictionary.csv')
impact_abbrev = pd.read_csv('../01_Notebooks/Data/impact_abbrev.csv')
model = pd.read_csv('../01_Notebooks/Data/model_2050.csv')

In [7]:
# Create a dict from the Programming Name and Long name columns of es_tech_dict
es_tech_df = es_tech_df[~es_tech_df['Programming name'].isin(wood_list+wet_biomass_list+waste_list)]  # keeping ES names for aggregation of biomass resources later on
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))

In [8]:
save_results = True
update_dat_files = False
dual = False

In [9]:
height=400
width=800

In [10]:
# Computation for ground-mounted PV potentiel
# surface [km2] * m2 of PV per m2 of land [-] * yield [-] * efficiency [-] * yearly insolation [GWh/km2/yr]
6300 * 0.4 * 0.88 * 0.1125 * (4.33 * 365) / 1e3 # TWh

394.290666

In [9]:
# Set up your Brightway project
bd.projects.set_current('ecoinvent3.10.1')

In [4]:
path_inputs = '../01_Notebooks/Data/'
path_model = '../02_AMPL_files/model/'
path_results = '../03_Results/'

In [11]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl'  # path to your AMPL license file
os.environ['PATH'] = path_to_ampl_licence + ':' + os.environ['PATH']

In [12]:
impact_abbrev.Impact_category = impact_abbrev.Impact_category.apply(lambda x: ast.literal_eval(x))

In [13]:
regionalized_hh_impact_categories = [i[-1] for i in impact_abbrev[(impact_abbrev.AoP == 'HH') & (impact_abbrev.Regionalized == True)].Impact_category.unique()]
regionalized_hh_impact_categories = [i for i in regionalized_hh_impact_categories if i not in ['Total human health', 'Remaining human health']]

In [14]:
regionalized_eq_impact_categories = [i[-1] for i in impact_abbrev[(impact_abbrev.AoP == 'EQ') & (impact_abbrev.Regionalized == True)].Impact_category.unique()]
regionalized_eq_impact_categories = [i for i in regionalized_eq_impact_categories if i not in ['Total ecosystem quality', 'Remaining ecosystem quality']]

## Running the optimization

In [15]:
if update_dat_files:

    main_db = Database([
        'ecoinvent_cutoff_3.10.1_image_SSP5-H_2023+truck_carculator',
        'ei_3.10.1_image_SSP5-H_2023+truck_carculator_reg',
        'ei_3.10.1_image_SSP5-H_2023+truck_carculator_reg_wo',
        'ecoinvent_cutoff_3.10.1_image_SSP5-H_2050_wo_updates+truck_carculator',
    ])

    for ssp_rcp in ['SSP2-L', 'SSP5-H']:

        # Set up your Brightway project
        bd.projects.set_current('ecoinvent3.10.1')

        main_db += Database([
            f'ecoinvent_cutoff_3.10.1_image_{ssp_rcp}_2050+truck_carculator',
            f'ei_3.10.1_image_{ssp_rcp}_2050+truck_carculator_reg',
            f'ei_3.10.1_image_{ssp_rcp}_2050+truck_carculator_reg_wo',
        ], create_pickle=True)

        update_ampl_files(
            year=2050,
            ssp_rcp=ssp_rcp,
            reg_level='all',
            specific_lcia_abbrev=['m_CCS_all', 'RHHD', 'REQD'],
            main_database=main_db,
        )

In [16]:
impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10')
    | (i[0] == 'IMPACT World+ Midpoint 2.1_regionalized for ecoinvent v3.10')
    | (i[0] == 'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)')
    | (i[0] == 'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)')
]

impact_categories_list +=[
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Total human health (biogenic)'),
]

impact_categories_list = [i for i in impact_categories_list if i not in [  # keep only marine acidification from -1/+1 version of IW+
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Marine acidification, short term'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Marine acidification, long term')
]]

In [ ]:
list_df_f_mult = []
list_df_annual_prod = []
list_df_annual_res = []
list_df_annual_prod_direct = []
list_total_cost = []
list_dual_variables = []
results_constraints = []
results_dict = {}

for ssp_rcp in ['SSP5-H', 'SSP2-L']:

    results_dict[ssp_rcp] = {}

    for reg_level in ['base_wo_iam', 'base', 'spat', 'spat_fore', 'spat_back', 'spat_fore_back']:

        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue

        t1 = time.time()

        try:
            results = run_opti(
                reg_level=reg_level,
                year=2050,
                ssp_rcp=ssp_rcp,
                other_emissions=True,
                constraint_on_remaining_aop=True,
                constraint_on_foreign_ghg_emissions=True,
                dual_variables=dual,
                returns='both' if dual else 'results'
            )
        except ValueError:
            print(f'{ssp_rcp}-{reg_level} is unfeasible')
            continue

        # Dual variables
        if dual:
            constraints_list = [
                'totalLCIA_limit', 'totalTERRITORIAL_limit', 'totalABROAD_limit', 'co2_emission', 'co2_emission2',  # environmental constraints
                'size_limit', 'prod_max', 'prod_min',  # technologies capacity potentials
                'resource_availability',  # availability of energy resources
                'f_min_perc', 'f_max_perc', 'f_min_perc_mob', 'f_max_perc_mob',
                'share_public_rail_sd_1', 'share_public_rail_sd_2', 'share_public_rail_md_1', 'share_public_rail_md_2', 'share_public_rail_ld_1', 'share_public_rail_ld_2', 'share_public_rail_eld_1', 'share_public_rail_eld_2', 'share_public_air_ld_1', 'share_public_air_ld_2', 'share_public_air_eld_1', 'share_public_air_eld_2',  # Passenger mobility share limits
                'share_freight_rail_ld_1', 'share_freight_rail_ld_2', 'share_freight_rail_eld_1', 'share_freight_rail_eld_2', 'share_freight_rail_ld_1', 'share_freight_rail_ld_2', 'share_freight_rail_eld_1', 'share_freight_rail_eld_2', # Freight mobility share limits
            ]
            es, results = results
            df_dual = pd.concat([es.es_model.get_constraint(constr_name).get_values().to_pandas().melt(ignore_index=False).reset_index().rename(columns={'index': 'index0', 'variable': 'Constraint', 'value': 'Dual value'}) for constr_name in constraints_list])
            df_dual = df_dual[df_dual['Dual value'] != 0]
            df_dual['Constraint'] = df_dual['Constraint'].apply(lambda x: x.replace('.dual', ''))
            df_dual['Run'] = f'{reg_level}-{ssp_rcp}'
            list_dual_variables.append(df_dual)

        max_indicator = pd.read_csv(f'../02_AMPL_files/data/2050/{reg_level}/{ssp_rcp}/QC_techs_lca_max.csv')
        max_ccs_norm = max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0]
        max_rhh_norm = max_indicator[max_indicator.Abbrev == 'RHHD'].max_unit.iloc[0]
        max_req_norm = max_indicator[max_indicator.Abbrev == 'REQD'].max_unit.iloc[0]

        # Territorial/abroad emissions
        total_ccst = results.variables['TotalLCIA']['TotalLCIA'].loc['m_CCS_all'] * max_ccs_norm
        territorial_ccst = results.variables['TotalTERRITORIAL']['TotalTERRITORIAL'].loc['m_CCS_all'] * max_ccs_norm
        limit_terr_ccst = results.parameters['limit_territorial']['limit_territorial'].loc['m_CCS_all'] * max_ccs_norm
        abroad_ccst = results.variables['TotalABROAD']['TotalABROAD'].loc['m_CCS_all'] * max_ccs_norm
        limit_abroad_ccst = results.parameters['limit_abroad']['limit_abroad'].loc['m_CCS_all'] * max_ccs_norm
        remaining_hh = results.variables['TotalLCIA']['TotalLCIA'].loc['RHHD'] * max_rhh_norm
        limit_rhh = results.parameters['limit_lcia']['limit_lcia'].loc['RHHD'] * max_rhh_norm
        remaining_eq = results.variables['TotalLCIA']['TotalLCIA'].loc['REQD'] * max_req_norm
        limit_req = results.parameters['limit_lcia']['limit_lcia'].loc['REQD'] * max_req_norm
        total_cost = results.variables['TotalCost']['TotalCost'].iloc[0]
        results_constraints.append([
            f'{reg_level}-{ssp_rcp}',
            total_ccst,
            territorial_ccst,
            limit_terr_ccst,
            abroad_ccst,
            limit_abroad_ccst,
            remaining_hh,
            limit_rhh,
            remaining_eq,
            limit_req,
            total_cost,
        ])
        results_dict[ssp_rcp][reg_level] = results

        if reg_level == 'base_wo_iam':
            impact_scores_2023 = pd.read_csv(path_results+f'LCA/2023/base/impact_scores.csv')
            impact_scores = pd.read_csv(path_results+f'LCA/2050/{reg_level}/SSP5-H/impact_scores.csv')
            impact_scores_direct = pd.read_csv(path_results+f'LCA/2050/{reg_level}/SSP5-H/impact_scores_direct_emissions.csv')
        else:
            impact_scores_2023 = pd.read_csv(path_results+f'LCA/2023/{reg_level}/impact_scores.csv')
            impact_scores = pd.read_csv(path_results+f'LCA/2050/{reg_level}/{ssp_rcp}/impact_scores.csv')
            impact_scores_direct = pd.read_csv(path_results+f'LCA/2050/{reg_level}/{ssp_rcp}/impact_scores_direct_emissions.csv')

        impact_scores = update_existing_infrastructure_metrics(
            df_impact_2050=impact_scores,
            df_impact_2020=impact_scores_2023,
            list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        )

        impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores_direct = add_biogenic_climate_change_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores_direct = add_rhhd_and_reqd_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores,
            df_results=results,
        )

        df_annual_prod_direct = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores_direct,
            df_results=results,
            assessment_type='direct',
        )

        all_mob_techs = (
            list(results.sets['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES'])
            + list(results.sets['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES'])
            + list(results.sets['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES'])
        )

        # Keeping mobility sub-models only
        df_f_mult = df_f_mult[~df_f_mult['index'].isin(all_mob_techs)]
        df_annual_prod = df_annual_prod[~df_annual_prod['index'].isin(all_mob_techs)]
        df_annual_prod_direct = df_annual_prod_direct[~df_annual_prod_direct['index'].isin(all_mob_techs)]

        df_annual_prod = df_annual_prod.merge(
            results.variables['ABROAD_op'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'ABROAD_op': 'Climate change, short term, total (abroad)'})
        df_annual_prod = df_annual_prod.merge(
            results.variables['TERRITORIAL_op'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'TERRITORIAL_op': 'Climate change, short term, total (territorial)'})
        df_f_mult = df_f_mult.merge(
            results.variables['ABROAD_constr'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'ABROAD_constr': 'Climate change, short term, total (abroad)'})
        df_f_mult = df_f_mult.merge(
            results.variables['TERRITORIAL_constr'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'TERRITORIAL_constr': 'Climate change, short term, total (territorial)'})
        df_annual_res = df_annual_res.merge(
            results.variables['ABROAD_res'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'ABROAD_res': 'Climate change, short term, total (abroad)'})
        df_annual_res = df_annual_res.merge(
            results.variables['TERRITORIAL_res'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
        ).rename(columns={'TERRITORIAL_res': 'Climate change, short term, total (territorial)'})

        df_f_mult['Run'] = f'{reg_level}-{ssp_rcp}'
        df_annual_prod['Run'] = f'{reg_level}-{ssp_rcp}'
        df_annual_res['Run'] = f'{reg_level}-{ssp_rcp}'
        df_annual_prod_direct['Run'] = f'{reg_level}-{ssp_rcp}'

        df_f_mult = df_f_mult[df_f_mult.F_Mult != 0]
        df_annual_prod = df_annual_prod[df_annual_prod.Annual_Prod != 0]
        df_annual_res = df_annual_res[df_annual_res.Annual_Res != 0]
        df_annual_prod_direct = df_annual_prod_direct[df_annual_prod_direct.Annual_Prod != 0]

        list_df_f_mult.append(df_f_mult)
        list_df_annual_prod.append(df_annual_prod)
        list_df_annual_res.append(df_annual_res)
        list_df_annual_prod_direct.append(df_annual_prod_direct)

        list_total_cost.append(results.variables['TotalCost'].TotalCost.iloc[0])

        fig_data = generate_sankey_flows(
                results=results,
                aggregate_mobility=True,
                aggregate_grid=True,
                aggregate_technology=True,
                run_id=0,
            )
        fig_data['source (long)'] = fig_data.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
        fig_data['target (long)'] = fig_data.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

        fig = _create_sankey_figure(fig_data, colors=default_colors_sankey, long_names=True)
        if save_results:
            fig.write_html(path_results + f'Figures/2050/sankey_{ssp_rcp}_{reg_level}.html')

        t2 = time.time()
        print(f'{ssp_rcp}-{reg_level} done in {round(t2-t1,0)} seconds!')

df_f_mult = pd.concat(list_df_f_mult)
df_annual_prod = pd.concat(list_df_annual_prod)
df_annual_res = pd.concat(list_df_annual_res)
df_annual_prod_direct = pd.concat(list_df_annual_prod_direct)

df_f_mult = df_f_mult.merge(model[model['Amount'] == 1], left_on='index', right_on='Name', how='left').rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])
df_annual_prod = df_annual_prod.merge(model[model['Amount'] == 1], left_on='index', right_on='Name', how='left').rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])
df_annual_prod_direct = df_annual_prod_direct.merge(model[model['Amount'] == 1], left_on='index', right_on='Name', how='left').rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])
df_f_mult['Main production'] = df_f_mult['Main production'].astype(str)
df_annual_prod['Main production'] = df_annual_prod['Main production'].astype(str)
df_annual_prod_direct['Main production'] = df_annual_prod_direct['Main production'].astype(str)


df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
df_annual_prod_direct['Sector'] = df_annual_prod_direct.apply(category_to_sector, axis=1)
df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
# df_annual_res['Sector'] = df_annual_res.apply(lambda x: 'Energy resources' if x['Climate change, short term, total'] >= 0 else 'Energy resources (CO2 uptake)', axis=1)
# df_annual_res['Category'] = df_annual_res.apply(lambda x: 'Energy resources' if x['Climate change, short term, total'] >= 0 else 'Energy resources (CO2 uptake)', axis=1)
df_annual_res['Sector'] = df_annual_res.apply(lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV' else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'), axis=1)
df_annual_res['Category'] = df_annual_res.apply(lambda x: 'ELECTRICITY_EHV' if x['index'] == 'ELECTRICITY_EHV' else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'), axis=1)

df_results_constraints = pd.DataFrame(
    data=results_constraints,
    columns=[
        'Run',
        'Total CC',
        'Territorial CC',
        'Limit terr CC',
        'Abroad CC',
        'Limit abroad CC',
        'Remaining HH',
        'Limit RHHD',
        'Remaining EQ',
        'Limit REQD',
        'Total cost',
    ]
)
df_results_constraints.Run = df_results_constraints.Run.apply(lambda x: reg_level_name_dict_2050[x])

if dual:
    df_dual = pd.concat(list_dual_variables)
    if save_results:
        df_dual.to_csv(path_results+'Tables/2050/df_dual.csv', index=False)

if save_results:
    df_f_mult.to_csv(path_results+'Tables/2050/df_f_mult.csv', index=False)
    df_annual_prod.to_csv(path_results+'Tables/2050/df_annual_prod.csv', index=False)
    df_annual_res.to_csv(path_results+'Tables/2050/df_annual_res.csv', index=False)
    df_annual_prod_direct.to_csv(path_results+'Tables/2050/df_annual_prod_direct.csv', index=False)
    df_results_constraints.to_csv(path_results+'Tables/2050/df_results_constraints.csv', index=False)

In [ ]:
df_results_constraints

In [ ]:
if dual:
    df_dual.head()

## Gather the results into dataframes

In [ ]:
# To skip previous steps
# df_f_mult = pd.read_csv(path_results+f'Tables/2050/df_f_mult.csv')
# df_annual_prod = pd.read_csv(path_results+f'Tables/2050/df_annual_prod.csv')
# df_annual_res = pd.read_csv(path_results+f'Tables/2050/df_annual_res.csv')
# df_annual_prod_direct = pd.read_csv(path_results+f'Tables/2050/df_annual_prod_direct.csv')
# df_results_constraints = pd.read_csv(path_results+f'Tables/2050/df_results_constraints.csv')

In [ ]:
df_annual_prod = aggregate_mobility_submodels(df_annual_prod)

In [ ]:
df_annual_prod_direct = aggregate_mobility_submodels(df_annual_prod_direct)

In [ ]:
df_f_mult = aggregate_mobility_submodels(df_f_mult)

In [ ]:
df_annual_prod['Run'] = df_annual_prod['Run'].replace(reg_level_name_dict_2050)
df_f_mult['Run'] = df_f_mult['Run'].replace(reg_level_name_dict_2050)
df_annual_res['Run'] = df_annual_res['Run'].replace(reg_level_name_dict_2050)
df_annual_prod_direct['Run'] = df_annual_prod_direct['Run'].replace(reg_level_name_dict_2050)

df_annual_prod['index'] = df_annual_prod['index'].replace(es_tech_name_dict)
df_f_mult['index'] = df_f_mult['index'].replace(es_tech_name_dict)
df_annual_res['index'] = df_annual_res['index'].replace(es_tech_name_dict)
df_annual_prod_direct['index'] = df_annual_prod_direct['index'].replace(es_tech_name_dict)

In [ ]:
df_annual_prod_elec = df_annual_prod[(df_annual_prod.Sector == 'Electricity') & ~(df_annual_prod['index'].str.endswith('Transformer'))]
df_annual_prod_dom_heat = df_annual_prod[(df_annual_prod.Sector == 'Domestic heat')]
df_annual_prod_ind_heat = df_annual_prod[(df_annual_prod.Sector == 'Industrial heat')]
df_annual_prod_all_heat = df_annual_prod[(df_annual_prod.Sector == 'Industrial heat') | (df_annual_prod.Sector == 'Domestic heat')]
df_annual_prod_pass_mob = df_annual_prod[(df_annual_prod.Sector == 'Passenger mobility')]
df_annual_prod_freight_mob = df_annual_prod[(df_annual_prod.Sector == 'Freight mobility')]
df_annual_prod_dac = df_annual_prod[(df_annual_prod.Sector == 'Carbon capture')]
df_annual_prod_other = df_annual_prod[(df_annual_prod.Sector == 'Other')].dropna(subset=['Total human health'])

In [ ]:
df_f_mult_elec = df_f_mult[(df_f_mult.Sector == 'Electricity') & ~(df_f_mult['index'].str.endswith('Transformer'))]
df_f_mult_dom_heat = df_f_mult[(df_f_mult.Sector == 'Domestic heat')]
df_f_mult_ind_heat = df_f_mult[(df_f_mult.Sector == 'Industrial heat')]
df_f_mult_all_heat = df_f_mult[(df_f_mult.Sector == 'Industrial heat') | (df_f_mult.Sector == 'Domestic heat')]
df_f_mult_pass_mob = df_f_mult[(df_f_mult.Sector == 'Passenger mobility')]
df_f_mult_freight_mob = df_f_mult[(df_f_mult.Sector == 'Freight mobility')]
df_f_mult_other = df_f_mult[(df_f_mult.Sector == 'Other')].dropna(subset=['Total human health'])

In [ ]:
df_annual_prod['Total human health (regionalized part)'] = 0
for col in list(df_annual_prod.columns):
    if col in regionalized_hh_impact_categories:
        df_annual_prod['Total human health (regionalized part)'] += df_annual_prod[col]

df_annual_prod_direct['Total human health (regionalized part)'] = 0
for col in list(df_annual_prod_direct.columns):
    if col in regionalized_hh_impact_categories:
        df_annual_prod_direct['Total human health (regionalized part)'] += df_annual_prod_direct[col]

df_annual_res['Total human health (regionalized part)'] = 0
for col in list(df_annual_res.columns):
    if col in regionalized_hh_impact_categories:
        df_annual_res['Total human health (regionalized part)'] += df_annual_res[col]

df_f_mult['Total human health (regionalized part)'] = 0
for col in list(df_f_mult.columns):
    if col in regionalized_hh_impact_categories:
        df_f_mult['Total human health (regionalized part)'] += df_f_mult[col]

In [ ]:
df_annual_prod['Total ecosystem quality (regionalized part)'] = 0
for col in list(df_annual_prod.columns):
    if col in regionalized_eq_impact_categories:
        df_annual_prod['Total ecosystem quality (regionalized part)'] += df_annual_prod[col]

df_annual_prod_direct['Total ecosystem quality (regionalized part)'] = 0
for col in list(df_annual_prod_direct.columns):
    if col in regionalized_eq_impact_categories:
        df_annual_prod_direct['Total ecosystem quality (regionalized part)'] += df_annual_prod_direct[col]

df_annual_res['Total ecosystem quality (regionalized part)'] = 0
for col in list(df_annual_res.columns):
    if col in regionalized_eq_impact_categories:
        df_annual_res['Total ecosystem quality (regionalized part)'] += df_annual_res[col]

df_f_mult['Total ecosystem quality (regionalized part)'] = 0
for col in list(df_f_mult.columns):
    if col in regionalized_eq_impact_categories:
        df_f_mult['Total ecosystem quality (regionalized part)'] += df_f_mult[col]

In [ ]:
def summary_df_cat(cat):

    if cat == 'Climate change, short term, total':
        conv_factor = 1e3
        uptake_cat = 'Climate change, short term, CO2 uptake'
    elif cat == 'Total human health (biogenic)':
        conv_factor = 1e6
        uptake_cat = ['Climate change, human health, short term, CO2 uptake', 'Climate change, human health, long term, CO2 uptake']
    elif cat == 'Total ecosystem quality (biogenic)':
        conv_factor = 1e6
        uptake_cat = ['Climate change, ecosystem quality, short term, CO2 uptake', 'Climate change, ecosystem quality, long term, CO2 uptake']
    else:
        raise ValueError('Unrecognized category')

    df_constr = df_f_mult.groupby(['Run']).agg({cat: 'sum'}) * conv_factor / N_capita_2050
    df_constr.rename(columns={cat: 'Construction'}, inplace=True)

    df_op = df_annual_prod.groupby(['Run']).agg({cat: 'sum'}) * conv_factor / N_capita_2050
    df_op.rename(columns={cat: 'Operation'}, inplace=True)

    df_op_direct = df_annual_prod_direct[df_annual_prod_direct.Sector != 'Carbon capture'].groupby(['Run']).agg({cat: 'sum'}) * conv_factor / N_capita_2050
    df_op_direct.rename(columns={cat: 'Operation (direct)'}, inplace=True)

    df_op_direct_capture = df_annual_prod_direct[df_annual_prod_direct.Sector == 'Carbon capture'].groupby(['Run']).agg({cat: 'sum'}) * conv_factor / N_capita_2050
    df_op_direct_capture.rename(columns={cat: 'Operation (carbon capture)'}, inplace=True)

    df_res = df_annual_res.groupby(['Run']).agg({cat: 'sum'}) * conv_factor / N_capita_2050
    df_res.rename(columns={cat: 'Resource'}, inplace=True)

    if cat == 'Climate change, short term, total':
        df_res_neg = df_annual_res.groupby(['Run']).agg({uptake_cat: 'sum'}) * conv_factor / N_capita_2050
        df_res_neg.rename(columns={uptake_cat: 'Resource (CO2 uptake)'}, inplace=True)
        df = pd.concat([df_constr, df_op, df_op_direct, df_op_direct_capture, df_res, df_res_neg], axis=1).fillna(0)
        df['Resource (rest)'] = df['Resource'] - df['Resource (CO2 uptake)']

    elif cat in ['Total human health (biogenic)', 'Total ecosystem quality (biogenic)']:
        df_res_neg_short = df_annual_res.groupby(['Run']).agg({uptake_cat[0]: 'sum'}) * conv_factor / N_capita_2050
        df_res_neg_short.rename(columns={uptake_cat[0]: 'Resource (CO2 uptake)'}, inplace=True)
        df_res_neg_long = df_annual_res.groupby(['Run']).agg({uptake_cat[1]: 'sum'}) * conv_factor / N_capita_2050
        df_res_neg_long.rename(columns={uptake_cat[1]: 'Resource (CO2 uptake)'}, inplace=True)
        df_res_neg = df_res_neg_short + df_res_neg_long
        df = pd.concat([df_constr, df_op, df_op_direct, df_op_direct_capture, df_res, df_res_neg], axis=1).fillna(0)
        df['Resource (rest)'] = df['Resource'] - df['Resource (CO2 uptake)']

    else:
        df = pd.concat([df_constr, df_op, df_op_direct, df_op_direct_capture, df_res], axis=1).fillna(0)

    df['Operation (indirect)'] = df['Operation'] - (df['Operation (direct)'] + df['Operation (carbon capture)'])
    df['Total'] = df['Construction'] + df['Operation'] + df['Resource']
    df = df.reset_index()
    df.rename(columns={'Run': 'Prospective-regionalization modeling level'}, inplace=True)

    if cat == 'Climate change, short term, total':
        df.drop(columns=['Resource'], inplace=True)

    return df

## CCST results

In [ ]:
cc_cat_name = 'Climate change, short term, total'

In [ ]:
df_ccst = summary_df_cat(cc_cat_name)

In [ ]:
df_ccst

In [ ]:
df_annual_prod_indirect = df_annual_prod.merge(df_annual_prod_direct, on=['index', 'Run', 'Sector', 'Category', 'Annual_Prod'], suffixes=('', ' (direct)'), how='left')
for col in list(df_annual_prod.columns):
    if col not in ['index', 'Run', 'Sector', 'Category', 'Annual_Prod', 'Phase', 'Main production']:
        if col in [cc_cat_name + ' (territorial)', cc_cat_name + ' (abroad)']:
            df_annual_prod_direct[col] = 0  # phases are won't be used for these two categories
        else:
            df_annual_prod_indirect[col] = df_annual_prod_indirect[col] - df_annual_prod_indirect[f"{col} (direct)"]
            df_annual_prod_indirect.drop(columns=[f"{col} (direct)"], inplace=True)

In [ ]:
cat_list = ['Total ecosystem quality (biogenic)', 'Total ecosystem quality (regionalized part)', 'Total human health (biogenic)', 'Total human health (regionalized part)', cc_cat_name, cc_cat_name+' (abroad)', cc_cat_name + ' (territorial)', 'Remaining human health', 'Remaining ecosystem quality']
col = ['Run', 'index', 'Sector', 'Phase'] + cat_list
df_f_mult['Phase'] = 'Construction'
df_annual_prod['Phase'] = 'Operation'
df_annual_prod_direct['Phase'] = 'Operation (direct)'
df_annual_prod_indirect['Phase'] = 'Operation (indirect)'
df_annual_res['Phase'] = 'Resource'
df_total_impact = pd.concat([
    df_f_mult[['F_Mult'] + col].rename(columns={'F_Mult': 'Capacity or production'}),
    df_annual_prod_direct[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_prod_indirect[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_res[['Annual_Res'] + col].rename(columns={'Annual_Res': 'Capacity or production'}),
],
    ignore_index=True)

In [ ]:
df_total_impact[cc_cat_name] *= 1e3 / N_capita_2050  # convert to t CO2 eq. per capita and per year
df_total_impact[cc_cat_name + ' (abroad)'] *= 1e3 / N_capita_2050
df_total_impact[cc_cat_name + ' (territorial)'] *= 1e3 / N_capita_2050

In [ ]:
if save_results:
    df_total_impact.to_csv(path_results+f'Tables/2050/df_total_impact.csv', index=False)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    show_direct_marker=False,
    show_total_marker=False,
    imp_cat=cc_cat_name,
    save_results=save_results,
    df_ccst_terr_abroad=df_results_constraints,
    hatch_phase=False,
    showlegend=True,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=cc_cat_name,
    save_results=save_results,
    group_by='index',
    cutoff=0.028,
    showlegend=True,
    df_ccst_terr_abroad=df_results_constraints,
    show_direct_marker=False,
    show_total_marker=False,
    hatch_phase=False,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=cc_cat_name + ' (abroad)',
    save_results=save_results,
    group_by='index',
    cutoff=0.025,
    showlegend=True,
    df_ccst_terr_abroad=df_results_constraints,
    show_direct_marker=False,
    show_total_marker=True,
    hatch_phase=False,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=cc_cat_name + ' (abroad)',
    save_results=save_results,
    showlegend=True,
    df_ccst_terr_abroad=df_results_constraints,
    show_direct_marker=False,
    show_total_marker=True,
    hatch_phase=False,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=cc_cat_name + ' (territorial)',
    save_results=save_results,
    group_by='index',
    cutoff=0.022,
    showlegend=True,
    df_ccst_terr_abroad=df_results_constraints,
    show_direct_marker=False,
    show_total_marker=True,
    hatch_phase=False,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=cc_cat_name + ' (territorial)',
    save_results=save_results,
    showlegend=True,
    df_ccst_terr_abroad=df_results_constraints,
    show_direct_marker=False,
    show_total_marker=True,
    hatch_phase=False,
)

## HH results

In [ ]:
df_hh = summary_df_cat('Total human health (biogenic)')

In [ ]:
df_hh

In [ ]:
df_total_impact['Total human health (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Total human health (regionalized part)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining human health'] *= 1e6 / N_capita_2050

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={'Total human health (biogenic)': 'Total human health'}),
    imp_cat='Total human health',
    save_results=save_results,
    show_direct_marker=False,
    show_regionalized_marker=False,
    showlegend=True,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={'Total human health (biogenic)': 'Total human health'}),
    imp_cat='Total human health',
    save_results=save_results,
    group_by='index',
    cutoff=0.03,
    showlegend=True,
    show_direct_marker=False,
    show_regionalized_marker=False,
)

In [ ]:
hh_impact_categories_list = [
    i[2] for i in impact_categories_list if
    (i[1] == 'Human health')
    & (i[2] not in ['Total human health', 'Total human health (biogenic)', 'Remaining human health'])
    & (i[2] not in [
        'Climate change, human health, short term',
        'Climate change, human health, short term, total',
        'Climate change, human health, long term',
        'Climate change, human health, long term, total'])
]
df_total_impact_cat_contrib_hh = pd.concat([
    df_f_mult[['Run', 'index', 'Sector', 'Phase'] + hh_impact_categories_list],
    df_annual_prod_direct[['Run', 'index', 'Sector', 'Phase'] + hh_impact_categories_list],
    df_annual_prod_indirect[['Run', 'index', 'Sector', 'Phase'] + hh_impact_categories_list],
    df_annual_res[['Run', 'index', 'Sector', 'Phase'] + hh_impact_categories_list],
], ignore_index=True)
df_total_impact_cat_contrib_hh = df_total_impact_cat_contrib_hh.melt(id_vars=['Run', 'index', 'Sector', 'Phase'], value_vars=hh_impact_categories_list, var_name='Impact category', value_name='Value')
df_total_impact_cat_contrib_hh['Regionalized'] = df_total_impact_cat_contrib_hh['Impact category'].apply(lambda x: True if x in regionalized_hh_impact_categories else False)

In [ ]:
df_total_impact_cat_contrib_hh['Value'] *= 1e6 / N_capita_2050
df_total_impact_cat_contrib_hh.rename(columns={'Value': 'Human health (biogenic)'}, inplace=True)

In [ ]:
if save_results:
    df_total_impact_cat_contrib_hh.to_csv(path_results+f'Tables/2050/df_contrib_imp_cat_tthh.csv', index=False)

In [ ]:
df_total_impact_cat_contrib_hh['Scope'] = df_total_impact_cat_contrib_hh['Phase'].apply(lambda x: 'Direct' if x == 'Operation (direct)' else 'Indirect')

In [ ]:
df_total_impact_cat_contrib_hh['Run'] = df_total_impact_cat_contrib_hh['Run'].apply(lambda x: x.replace('+', '\n'))

In [ ]:
plot_impact_categories_contribution(
    df_total_impact_cat_contrib_hh,
    'Human health (biogenic)',
    year=2050,
    cutoff=0.05,
    save_results=save_results,
    # df_results_constraints=df_results_constraints,
    show_direct_emissions_markers=False,
    show_regionalized_impact_markers=False,
    separate_negative_bars=True,
)

In [ ]:
df_rhh_constr = df_f_mult.groupby(['Run']).agg({'Remaining human health': 'sum'}) * 1e6 / N_capita_2050
df_rhh_constr.rename(columns={'Remaining human health': 'Construction'}, inplace=True)

df_rhh_op = df_annual_prod.groupby(['Run']).agg({'Remaining human health': 'sum'}) * 1e6 / N_capita_2050
df_rhh_op.rename(columns={'Remaining human health': 'Operation'}, inplace=True)

df_rhh_op_direct = df_annual_prod_direct.groupby(['Run']).agg({'Remaining human health': 'sum'}) * 1e6 / N_capita_2050
df_rhh_op_direct.rename(columns={'Remaining human health': 'Operation (direct)'}, inplace=True)

df_rhh_res = df_annual_res.groupby(['Run']).agg({'Remaining human health': 'sum'}) * 1e6 / N_capita_2050
df_rhh_res.rename(columns={'Remaining human health': 'Resource'}, inplace=True)

df_rhh = pd.concat([df_rhh_constr, df_rhh_op, df_rhh_op_direct, df_rhh_res], axis=1)
df_rhh['Operation (indirect)'] = df_rhh['Operation'] - df_rhh['Operation (direct)']
df_rhh['Total'] = df_rhh['Construction'] + df_rhh['Operation'] + df_rhh['Resource']

In [ ]:
df_rhh

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat='Remaining human health',
    save_results=save_results,
    group_by='index',
    cutoff=0.025,
    showlegend=True,
    show_direct_marker=False,
    show_total_marker=False,
    df_results_constraints=df_results_constraints,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat='Remaining human health',
    save_results=save_results,
    showlegend=True,
    show_direct_marker=False,
    show_total_marker=False,
    df_results_constraints=df_results_constraints,
)

## EQ results

In [ ]:
df_eq = summary_df_cat('Total ecosystem quality (biogenic)')

In [ ]:
df_eq

In [ ]:
df_total_impact['Total ecosystem quality (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Total ecosystem quality (regionalized part)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining ecosystem quality'] *= 1e6 / N_capita_2050

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={'Total ecosystem quality (biogenic)': 'Total ecosystem quality'}),
    imp_cat='Total ecosystem quality',
    save_results=save_results,
    showlegend=True,
    show_direct_marker=False,
    show_regionalized_marker=False,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={'Total ecosystem quality (biogenic)': 'Total ecosystem quality'}),
    imp_cat='Total ecosystem quality',
    save_results=save_results,
    group_by='index',
    cutoff=0.024,
    showlegend=True,
    show_direct_marker=False,
    show_regionalized_marker=False,
)

In [ ]:
eq_impact_categories_list = [
    i[2] for i in impact_categories_list if
    (i[1] == 'Ecosystem quality')
    & (i[2] not in ['Total ecosystem quality', 'Remaining ecosystem quality', 'Total ecosystem quality (biogenic)'])
    & (i[2] not in [
        'Climate change, ecosystem quality, short term',
        'Climate change, ecosystem quality, short term, total',
        'Climate change, ecosystem quality, long term',
        'Climate change, ecosystem quality, long term, total'])
]
df_total_impact_cat_contrib_eq = pd.concat([
    df_f_mult[['Run', 'index', 'Sector', 'Phase'] + eq_impact_categories_list],
    df_annual_prod_indirect[['Run', 'index', 'Sector', 'Phase'] + eq_impact_categories_list],
    df_annual_prod_direct[['Run', 'index', 'Sector', 'Phase'] + eq_impact_categories_list],
    df_annual_res[['Run', 'index', 'Sector', 'Phase'] + eq_impact_categories_list],
], ignore_index=True)
df_total_impact_cat_contrib_eq = df_total_impact_cat_contrib_eq.melt(id_vars=['Run', 'index', 'Sector', 'Phase'], value_vars=eq_impact_categories_list, var_name='Impact category', value_name='Value')
df_total_impact_cat_contrib_eq['Regionalized'] = df_total_impact_cat_contrib_eq['Impact category'].apply(lambda x: True if x in regionalized_eq_impact_categories else False)

In [ ]:
df_total_impact_cat_contrib_eq['Value'] *= 1e6 / N_capita_2050
df_total_impact_cat_contrib_eq.rename(columns={'Value': 'Ecosystem quality (biogenic)'}, inplace=True)

In [ ]:
if save_results:
    df_total_impact_cat_contrib_eq.to_csv(path_results+f'Tables/2050/df_contrib_imp_cat_tteq.csv', index=False)

In [ ]:
df_total_impact_cat_contrib_eq['Scope'] = df_total_impact_cat_contrib_eq['Phase'].apply(
    lambda x: 'Direct' if x == 'Operation (direct)' else 'Indirect')

In [ ]:
df_total_impact_cat_contrib_eq['Run'] = df_total_impact_cat_contrib_eq['Run'].apply(lambda x: x.replace('+', '\n'))

In [ ]:
plot_impact_categories_contribution(
    df_total_impact_cat_contrib_eq,
    'Ecosystem quality (biogenic)',
    year=2050,
    cutoff=0.03,
    save_results=save_results,
    # df_results_constraints=df_results_constraints,
    show_direct_emissions_markers=False,
    show_regionalized_impact_markers=False,
    separate_negative_bars=True,
)

In [ ]:
df_req_constr = df_f_mult.groupby(['Run']).agg({'Remaining ecosystem quality': 'sum'}) * 1e6 / N_capita_2050
df_req_constr.rename(columns={'Remaining ecosystem quality': 'Construction'}, inplace=True)

df_req_op = df_annual_prod.groupby(['Run']).agg({'Remaining ecosystem quality': 'sum'}) * 1e6 / N_capita_2050
df_req_op.rename(columns={'Remaining ecosystem quality': 'Operation'}, inplace=True)

df_req_op_direct = df_annual_prod_direct.groupby(['Run']).agg({'Remaining ecosystem quality': 'sum'}) * 1e6 / N_capita_2050
df_req_op_direct.rename(columns={'Remaining ecosystem quality': 'Operation (direct)'}, inplace=True)

df_req_res = df_annual_res.groupby(['Run']).agg({'Remaining ecosystem quality': 'sum'}) * 1e6 / N_capita_2050
df_req_res.rename(columns={'Remaining ecosystem quality': 'Resource'}, inplace=True)

df_req = pd.concat([df_req_constr, df_req_op, df_req_op_direct, df_req_res], axis=1)
df_req['Operation (indirect)'] = df_req['Operation'] - df_req['Operation (direct)']
df_req['Total'] = df_req['Construction'] + df_req['Operation'] + df_req['Resource']

In [ ]:
df_req

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat='Remaining ecosystem quality',
    save_results=save_results,
    group_by='index',
    cutoff=0.034,
    showlegend=True,
    show_direct_marker=False,
    df_results_constraints=df_results_constraints,
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat='Remaining ecosystem quality',
    save_results=save_results,
    showlegend=True,
    show_direct_marker=False,
    df_results_constraints=df_results_constraints,
)

## Energy system configuration

In [ ]:
show_delta = False

In [ ]:
df_annual_res_2023 = pd.read_csv('../03_Results/Tables/reference/df_annual_res.csv')
df_annual_prod_2023 = pd.read_csv('../03_Results/Tables/reference/df_annual_prod.csv')
df_f_mult_2023 = pd.read_csv('../03_Results/Tables/reference/df_f_mult.csv')

In [ ]:
df_annual_prod_2023 = aggregate_mobility_submodels(df_annual_prod_2023)
df_annual_prod_2023['index'] = df_annual_prod_2023['index'].replace(es_tech_name_dict)
df_f_mult_2023 = aggregate_mobility_submodels(df_f_mult_2023)
df_f_mult_2023['index'] = df_f_mult_2023['index'].replace(es_tech_name_dict)

In [ ]:
df_res = df_annual_res[(df_annual_res.Annual_Res > 0.1) & (df_annual_res['index'] != 'CO2_E')]
df_res = df_res.apply(lambda row: rename_bio_resources(row, name_type='long'), axis=1)
df_res.Run = df_res.Run.apply(lambda x: x.replace("+", " "))
df_res['Annual_Res'] *= 1e-3  # from GWh to TWh
df_res = df_res[~((df_res['index'].str.startswith('RES_')) | (df_res['index'].str.contains('potential')))].groupby(['Run', 'index']).sum().reset_index().sort_values('Annual_Res', ascending=False)
df_res['Run_hover'] = df_res['Run'].apply(lambda x: x.replace('Def.  SSP5-H', 'Default'))

df_res_2023 = df_annual_res_2023[df_annual_res_2023['Run'] == "base"]
df_res_2023 = df_res_2023[~df_res_2023['index'].str.startswith('RES_')]
df_res_2023['Annual_Res'] *= 1e-3  # from GWh to TWh
total_res_2023 = float(df_res_2023['Annual_Res'].sum())
df_res['Sector'] = 'Energy resources'
df_res['Unit'] = 'TWh/year'

if show_delta:
    # Get the reference data (default scenario)
    df_def = df_res[df_res['Run'] == 'Def.  SSP5-H'][['index', 'Run', 'Annual_Res']].copy()
    total_res_def = float(df_def['Annual_Res'].sum())

    # Ensure every Name in the reference has an entry for every Run in df_res
    for name in df_def['index'].unique():
        for run in df_res['Run'].unique():
            if run != 'Def.  SSP5-H':  # Skip the default run
                if len(df_res[(df_res['index'] == name) & (df_res['Run'] == run)]) == 0:
                    # Add a row with Annual_Res=0
                    new_row = {'index': name, 'Run': run, 'Annual_Res': 0}
                    df_res = pd.concat([df_res, pd.DataFrame([new_row])], ignore_index=True)

    df_res = df_res.merge(df_res[df_res['Run'] == 'Def.  SSP5-H'][['index', 'Run', 'Annual_Res']], how='outer', on=['index'], suffixes=('', '_def'))
    df_res['Annual_Res_def'] = df_res['Annual_Res_def'].fillna(0)
    df_res['Annual_Res'] = df_res['Annual_Res'].fillna(0)
    df_res['Delta'] = df_res['Annual_Res'] - df_res['Annual_Res_def']
    df_res['Delta_perc'] = 100 * df_res['Delta'] / total_res_def
    df_res = df_res[df_res['Run'] != 'Def.  SSP5-H']

hover_vars = ['Run_hover', 'index', 'Annual_Res']
fig = px.bar(
    df_res,
    y='Run',
    x='Annual_Res' if not show_delta else 'Delta_perc',
    orientation='h',
    color='index',
    color_discrete_map=techs_color_map,
    width=600,
    height=370,
    category_orders={
        'Run': [i.replace('\n', ' ') for i in run_order_2050],
    },
    labels={
        'Run': 'Prospective-regionalization level',
        'Annual_Res': 'Energy resources (TWh/year)',
        'index': 'Resource',
        'Delta': 'Difference with default in energy resources (TWh/year)',
        'Delta_perc': f'Difference with default in energy resources (%)',
    },
    hover_data=hover_vars,
)
hover_text = (
    f"<b>Prosp.-reg. level:</b> %{{customdata[0]}}<br>"
    f"<b>Resource:</b> %{{customdata[1]}}<br>"
)

hover_text += f"<b>Value:</b> %{{x:,.2f}} {'TWh / year' if not show_delta else '%'}<extra></extra>"

fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
    # legend_traceorder="reversed",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.25,
        xanchor="center",
        x=0.4,
    )
)

fig.update_traces(
    width=.6,
    hovertemplate=hover_text,
)

fig.add_hline(y=4.5, line_width=1, line_dash="dash", line_color="black")
if not show_delta:
    fig.add_hline(y=9.5, line_width=1, line_dash="dash", line_color="black")

fig.add_annotation(
    text="SSP5-H",
    xref="paper",
    yref="paper",
    textangle=-90,
    y=9.5/11 if not show_delta else 10/11,
    x=0,
    xshift=-140,
    showarrow=False,
    font=dict(size=15, color="black")
)

fig.add_annotation(
    text="SSP2-L",
    xref="paper",
    yref="paper",
    textangle=-90,
    y=1/11,
    x=0,
    xshift=-140,
    showarrow=False,
    font=dict(size=15, color="black")
)

y_vals = [i.replace('\n', ' ') for i in run_order_2050]

y_text = [
    r.replace(" SSP5-H", "")
    .replace(" SSP2-L", "")
    for r in y_vals
]

fig.update_yaxes(
    tickmode="array",
    tickvals=y_vals,
    ticktext=y_text,
)

if not show_delta:
    fig.add_vline(
        x=total_res_2023,
        line_dash="dot",
        line_color="red",
        line_width=3.5,
        opacity=0.9,
        label=dict(text="2023", font=dict(color="red", size=17))
    )

fig.show()

if save_results:
    fig.write_image(path_results + f'Figures/2050/annual_res{"_delta" if show_delta else ""}.pdf')
    fig.update_layout(width=None, height=None)
    fig.write_html(path_results + f'Figures/2050/annual_res{"_delta" if show_delta else ""}.html')

In [ ]:
df_res[df_res['index'] == 'Natural gas'][['Run', 'Annual_Res']]

In [ ]:
for distance_level in ['_SD', '_MD', '_LD', '_ELD']:
    model['Name'] = model['Name'].str.replace(distance_level, '')
    model['Flow'] = model['Flow'].str.replace(distance_level, '')
model.drop_duplicates(inplace=True)

model['Name'] = model['Name'].replace(es_tech_name_dict)

In [ ]:
df_elec = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    cutoff=0.02,
    annual_prod_2023=df_annual_prod_2023,
    sector='Electricity',
    save_results=save_results,
    return_df=True,
    show_delta=show_delta,
)

In [ ]:
plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    cutoff=0.02,
    sector='Electricity',
    save_results=save_results,
    return_df=False,
    show_delta=show_delta,
    fm_unit=True,
)

In [ ]:
df_elec[~df_elec['index'].str.startswith('Existing ')].groupby('Run')['Production'].sum()

In [ ]:
df_elec_cap = plot_configuration_sector(
    model=model,
    f_mult=df_f_mult,
    f_mult_2023=df_f_mult_2023,
    cutoff=0.02,
    sector='Electricity',
    save_results=save_results,
    show_delta=show_delta,
    return_df=True,
)

In [ ]:
df_storage_cap = plot_configuration_sector(
    model=model,
    f_mult=df_f_mult,
    f_mult_2023=df_f_mult_2023,
    sector='Storage',
    save_results=save_results,
    show_delta=show_delta,
    return_df=True,
)

In [ ]:
df_heat = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Heat',
    save_results=save_results,
    cutoff=0.015,
    show_delta=show_delta,
    return_df=True,
)

In [ ]:
df_heat[df_heat['Name'] == 'Industrial Direct Electricity Usage'][['Name', 'Run', 'Production', 'Production share']]

In [ ]:
df_freight_mob = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Freight mobility',
    save_results=save_results,
    cutoff=0.02,
    return_df=True,
    show_delta=show_delta,
)

In [ ]:
trucks_total_prod = df_freight_mob[df_freight_mob['index'].str.endswith(' truck')].groupby('Run').sum()['Production'].loc['Def.  SSP5-H']

df_freight_mob[
    (df_freight_mob['index'] == 'Diesel Hybrid Short Distance Semi-trailer truck')
].groupby('Run').sum()['Production'] / trucks_total_prod

In [ ]:
df_pass_mob = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Passenger mobility',
    save_results=save_results,
    cutoff=0.0,
    return_df=True,
    show_delta=show_delta,
)

In [ ]:
suv_and_cars_total_prod = df_pass_mob[
    (df_pass_mob['index'].str.contains('Sport Utility Vehicle'))
    | (df_pass_mob['index'].str.contains(' Car'))
].groupby('Run').sum()['Production'].loc['Def.  SSP5-H']

df_pass_mob[
    (df_pass_mob['index'].str.contains('Electric-Powered Sport Utility Vehicle'))
    | (df_pass_mob['index'].str.contains('Electric-Powered Car'))
].groupby('Run').sum()['Production'] / suv_and_cars_total_prod

In [ ]:
plane_total_prod = df_pass_mob[df_pass_mob['index'].str.contains('Passenger Plane')].groupby('Run').sum()['Production'].loc['Def.  SSP5-H']

df_pass_mob[
    (df_pass_mob['index'] == 'Bio-Jet Fuel-Powered Passenger Plane')
].groupby('Run').sum()['Production'] / plane_total_prod

In [ ]:
df_cc = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Carbon capture',
    save_results=save_results,
    cutoff=0,
    return_df=True,
    show_delta=show_delta,
)

In [ ]:
df_cc.groupby('Run').sum()['Production']

In [ ]:
df_seq_uti = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Carbon storage and utilization',
    save_results=save_results,
    cutoff=0,
    show_delta=show_delta,
    return_df=True,
)

In [ ]:
df_seq_uti[df_seq_uti.Name.isin(['Carbon Mineralization', 'Carbon Transport and Injection'])].groupby('Run').sum()['Production']

In [ ]:
df_fuels = plot_configuration_sector(
    model=model,
    annual_prod=df_annual_prod,
    annual_prod_2023=df_annual_prod_2023,
    sector='Other',
    save_results=save_results,
    cutoff=0.02,
    return_df=True,
    show_delta=show_delta,
)

In [ ]:
df_fuels.groupby('Run').sum()['Production']

In [ ]:
df_config = pd.concat([
    df_elec[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_heat[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_pass_mob[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_freight_mob[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_fuels[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_cc[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_seq_uti[['Run', 'Name', 'Sector', 'Production', 'Unit']],
    df_res[['Run', 'index', 'Sector', 'Annual_Res', 'Unit']].rename(columns={'index': 'Name', 'Annual_Res': 'Production'}),
]).rename(columns={'Production': 'Production (or imports for resources)', 'Name': 'Technology or resource'})

if save_results:
    df_config.to_csv(path_results+'Tables/2050/df_config.csv', index=False)

## Carbon balance

In [ ]:
for run in df_total_impact.Run.unique():
    plot_sankey_carbon_flows(
        run=run,
        df_total_impact=df_total_impact,
        model=model,
        cutoff=0,
        aggregate_technologies=True,
        show_figure=False,
        save_results=save_results,
        per_capita=False,
        mode='mfa',
    )

In [ ]:
df = plot_sankey_carbon_flows(
    run=run,
    df_total_impact=df_total_impact,
    model=model,
    cutoff=0,
    aggregate_technologies=True,
    show_figure=True,
    save_results=False,
    per_capita=False,
    mode='mfa',
    return_df=True,
)

## Quantifying the "error" across configurations

In [20]:
df_f_mult = pd.read_csv('../03_Results/Tables/2050/df_f_mult.csv')
df_annual_prod = pd.read_csv('../03_Results/Tables/2050/df_annual_prod.csv')
df_annual_res = pd.read_csv('../03_Results/Tables/2050/df_annual_res.csv')
cat_list = [
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
]

df_total = pd.DataFrame(columns=['Run', 'Impact category', 'value'])

for ssp_rcp in ['SSP5-H', 'SSP2-L']:
    for reg_level in ['base_wo_iam', 'base', 'spat', 'spat_fore', 'spat_back', 'spat_fore_back']:
        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue
        impact_scores_2023 = pd.read_csv(path_results+f'LCA/2023/{reg_level if reg_level != 'base_wo_iam' else 'base'}/impact_scores.csv')
        impact_scores = pd.read_csv(path_results+f'LCA/2050/{reg_level}/{ssp_rcp}/impact_scores.csv')
        impact_scores = update_existing_infrastructure_metrics(
            df_impact_2050=impact_scores,
            df_impact_2020=impact_scores_2023,
            list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        )
        impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores = impact_scores.pivot_table(index=['Name', 'Type'], columns='Impact_category', values='Value')[[str(cat) for cat in cat_list]].reset_index()
        impact_scores = impact_scores.rename(columns={str(cat): cat[-1] + f' - {reg_level}-{ssp_rcp}' for cat in cat_list})

        df_f_mult = pd.merge(
            df_f_mult[['index', 'Run', 'F_Mult', 'lifetime']],
            impact_scores[impact_scores['Type'] == 'Construction'],
            how='left',
            left_on='index',
            right_on='Name',
        ).drop(columns=['Name'])

        df_annual_prod = pd.merge(
            df_annual_prod[['index', 'Run', 'Annual_Prod']],
            impact_scores[impact_scores['Type'] == 'Operation'],
            how='left',
            left_on='index',
            right_on='Name',
        ).drop(columns=['Name'])

        df_annual_res = pd.merge(
            df_annual_res[['index', 'Run', 'Annual_Res']],
            impact_scores[impact_scores['Type'] == 'Resource'],
            how='left',
            left_on='index',
            right_on='Name',
        ).drop(columns=['Name'])

        for cat in cat_list:
            df_f_mult[cat[-1] + f' - {reg_level}-{ssp_rcp}'] *= df_f_mult['F_Mult'] / df_f_mult['lifetime']
            df_annual_prod[cat[-1] + f' - {reg_level}-{ssp_rcp}'] *= df_annual_prod['Annual_Prod']
            df_annual_res[cat[-1] + f' - {reg_level}-{ssp_rcp}'] *= df_annual_res['Annual_Res']

        cols = ['index', 'Run'] + [cat[-1] + f' - {reg_level}-{ssp_rcp}' for cat in cat_list]
        df_total_level = pd.concat([
            df_f_mult[cols],
            df_annual_prod[cols],
            df_annual_res[cols],
        ])

        df_total_level = df_total_level.groupby(['Run']).sum().reset_index().melt(
            id_vars='Run',
            value_vars=[cat[-1] + f' - {reg_level}-{ssp_rcp}' for cat in cat_list],
            var_name='Impact category',
        )

        df_total = pd.concat([df_total, df_total_level])

df_total['Assessment level'] = df_total['Impact category'].apply(lambda x: x.split(' - ')[-1])
df_total['Impact category'] = df_total['Impact category'].apply(lambda x: x.split(' - ')[0])
# df_total['value'] *= 1e6 / N_capita_2050

df_total = df_total.merge(
    df_total[df_total['Assessment level'] == df_total['Run']][['Run', 'Impact category', 'value']],
    how='left',
    on=['Run', 'Impact category'],
    suffixes=('', ' reference'),
)

df_total['Delta_perc'] = 100 * (df_total['value'] - df_total['value reference']) / df_total['value reference']

df_total['Run'] = df_total['Run'].apply(lambda x: reg_level_name_dict_2050[x].replace('+', ' '))
df_total['Assessment level'] = df_total['Assessment level'].apply(lambda x: reg_level_name_dict_2050[x].replace('+', ' '))
df_total['Regionalization level'] = df_total['Assessment level'].apply(lambda x: x.split('  ')[0])
df_total['SSP-RCP'] = df_total['Assessment level'].apply(lambda x: x.split('  ')[-1])

C:\Users\matth\AppData\Local\Temp\ipykernel_22756\1738699318.py:69: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [36]:
import plotly.graph_objects as go
import plotly.express as px

df_plot = df_total[df_total['Assessment level'] != df_total['Run']].copy()

run_order_x = [i.replace('\n', ' ') for i in run_order_2050]

impact_categories = df_plot['Impact category'].unique()
regio_levels = df_plot['Regionalization level'].unique()   # e.g. Def., IAM, IAM Spat., ...
ssp_rcp_levels = df_plot['SSP-RCP'].unique()                # SSP5-H, SSP2-L

color_seq = px.colors.qualitative.Plotly
color_map = {cat: color_seq[i % len(color_seq)] for i, cat in enumerate(impact_categories)}

symbol_seq = ['circle', 'square', 'diamond', 'x', 'triangle-up', 'star', 'pentagon']
symbol_map = {lvl: symbol_seq[i % len(symbol_seq)] for i, lvl in enumerate(regio_levels)}

outline_map = {
    'SSP5-H': dict(width=1.5, color='black'),
    'SSP2-L': dict(width=0, color='rgba(0,0,0,0)')
}

fig = go.Figure()

# real data traces
for cat in impact_categories:
    for rlvl in regio_levels:
        for ssp in ssp_rcp_levels:
            sub = df_plot[
                (df_plot['Impact category'] == cat) &
                (df_plot['Regionalization level'] == rlvl) &
                (df_plot['SSP-RCP'] == ssp)
            ]
            if sub.empty:
                continue
            fig.add_trace(go.Scatter(
                x=sub['Delta_perc'], y=sub['Run'], mode='markers',
                marker=dict(
                    color=color_map[cat], symbol=symbol_map[rlvl], size=9,
                    opacity=0.6, line=outline_map[ssp]
                ),
                showlegend=False,
                customdata=sub[['Impact category', 'Regionalization level', 'SSP-RCP']],
                hovertemplate=(
                    "%{customdata[0]}<br>"
                    "Regionalization: %{customdata[1]}<br>"
                    "SSP-RCP: %{customdata[2]}<br>"
                    "%{x:.2f}%<extra></extra>"
                )
            ))

# dummy traces -> legend 1: color (impact category)
for cat in impact_categories:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(color=color_map[cat], size=10, symbol='circle'),
        name=cat, legend='legend', showlegend=True
    ))

# dummy traces -> legend 2: symbol (regionalization level)
for rlvl in regio_levels:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(color='gray', size=10, symbol=symbol_map[rlvl]),
        name=rlvl, legend='legend2', showlegend=True
    ))

# dummy traces -> legend 3: outline (SSP-RCP)
for ssp in ssp_rcp_levels:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(color='lightgray', size=10, symbol='circle', line=outline_map[ssp]),
        name=ssp, legend='legend3', showlegend=True
    ))

fig.update_yaxes(type='category', categoryorder='array', categoryarray=run_order_x, autorange='reversed')

fig.add_hline(y=0.5, line_width=1, line_dash="dash", line_color="black")
fig.add_hline(y=5.5, line_width=1, line_dash="dash", line_color="black")

fig.add_annotation(text="SSP5-H", xref="paper", yref="paper", textangle=-90,
                    y=9.2/11, x=0, xshift=-140, showarrow=False, font=dict(size=15, color="black"))
fig.add_annotation(text="SSP2-L", xref="paper", yref="paper", textangle=-90,
                    y=1.5/11, x=0, xshift=-140, showarrow=False, font=dict(size=15, color="black"))

x_text = [r.replace(" SSP5-H", "").replace(" SSP2-L", "") for r in run_order_x]
fig.update_yaxes(tickmode="array", tickvals=run_order_x, ticktext=x_text)

fig.update_layout(
    height=400,
    width=800,
    xaxis=dict(title=dict(text='Relative difference with respect to level (%)', standoff=20)),
    yaxis_title='Prospective-regionalization level',
    legend=dict(title='Impact category', orientation='h',
                yanchor='top', y=-0.25, xanchor='left', x=0),
    legend2=dict(title='Regionalization level', orientation='h',
                 yanchor='top', y=-0.35, xanchor='left', x=0),
    legend3=dict(title='SSP-RCP (black outline = SSP5-H)', orientation='h',
                 yanchor='top', y=-0.55, xanchor='left', x=0),
    margin=dict(l=5, r=5, t=5, b=5)
)

fig.show()

if save_results:
    fig.write_html('../03_Results/Figures/2050/relative_comparison_burden_shifts.html')
    fig.write_image('../03_Results/Figures/2050/relative_comparison_burden_shifts.pdf')